In [2]:
%pip install "numpy<2.0"

  Using cached numpy-1.26.4-cp310-cp310-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp310-cp310-win_amd64.whl (15.8 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.


In [1]:
import os
import shutil
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix
import warnings

warnings.filterwarnings('ignore')

# Настройка GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except RuntimeError as e:
        print(e)

BASE_DIR = "data"
CLASSES = ['0', '1', '3', '8']
NUM_CLASSES = len(CLASSES)

def prepare_data_splits():
    """Разделяет исходные данные на указанные пропорции для каждого сценария."""
    splits_config = {
        '100_15_20': (100, 15, 20),
        '250_30_20': (250, 30, 20),
        '400_60_20': (400, 60, 20),
        '700_100_20': (700, 80, 20)
    }
    
    src_train = os.path.join(BASE_DIR, 'train')
    
    for split_name, (n_train, n_val, n_test) in splits_config.items():
        split_base = os.path.join(BASE_DIR, f'split_{split_name}')
        for part in ['train', 'val', 'test']:
            for cls in CLASSES:
                os.makedirs(os.path.join(split_base, part, cls), exist_ok=True)
                
        for cls in CLASSES:
            cls_dir = os.path.join(src_train, cls)
            files = [f for f in os.listdir(cls_dir) if f.endswith('.png')]
            random.shuffle(files)
            
            selected = files[:n_train + n_val + n_test]
            partitions = {
                'train': selected[:n_train],
                'val': selected[n_train:n_train + n_val],
                'test': selected[n_train + n_val:]
            }
            
            for part, imgs in partitions.items():
                for img in imgs:
                    src = os.path.join(cls_dir, img)
                    dst = os.path.join(split_base, part, cls, img)
                    if not os.path.exists(dst):
                        shutil.copy(src, dst)

def get_model_config(model_name):
    """Возвращает архитектуру и требуемый размер входа для модели."""
    sizes = {
        'Xception': (224, 224),
        'ResNet50V2': (224, 224),
        'InceptionResNetV2': (139, 139),
        'DenseNet201': (224, 224),
        'NASNetLarge': (331, 331)
    }
    base_model = getattr(keras.applications, model_name)(
        weights='imagenet',
        include_top=False,
        input_shape=(sizes[model_name][0], sizes[model_name][1], 3)
    )
    base_model.trainable = False
    return base_model, sizes[model_name]

def build_model(model_name):
    base_model, input_size = get_model_config(model_name)
    
    inputs = keras.Input(shape=(input_size[0], input_size[1], 3))
    
    # Аугментация на GPU
    x = layers.RandomFlip("horizontal")(inputs)
    x = layers.RandomRotation(0.1)(x)
    x = layers.RandomTranslation(0.1, 0.1)(x)
    
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    
    model = keras.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model, input_size

def run_training(model_name, split_name, epochs_frozen=5, epochs_finetune=5, batch_size=16):
    """Обучает модель, оценивает на тесте и возвращает метрики."""
    split_dir = os.path.join(BASE_DIR, f'split_{split_name}')
    model, img_size = build_model(model_name)
    
    ds_kwargs = dict(image_size=img_size, batch_size=batch_size, label_mode='categorical')
    train_ds = tf.keras.utils.image_dataset_from_directory(os.path.join(split_dir, 'train'), **ds_kwargs, shuffle=True)
    val_ds = tf.keras.utils.image_dataset_from_directory(os.path.join(split_dir, 'val'), **ds_kwargs, shuffle=False)
    test_ds = tf.keras.utils.image_dataset_from_directory(os.path.join(split_dir, 'test'), **ds_kwargs, shuffle=False)
    
    # Оптимизация пайплайна данных
    autotune = tf.data.AUTOTUNE
    train_ds = train_ds.cache().prefetch(autotune)
    val_ds = val_ds.cache().prefetch(autotune)
    test_ds = test_ds.cache().prefetch(autotune)
    
    callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint(f'weights_{model_name}_{split_name}.keras', 
                                    monitor='val_loss', save_best_only=True, mode='min'), # 🔹 Унифицировали метрику
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-6, verbose=1)
    ]
    
    model.fit(train_ds, validation_data=val_ds, epochs=epochs_frozen, callbacks=callbacks, verbose=1)
    
    base_model, _ = get_model_config(model_name)
    
    for layer in base_model.layers[-int(len(base_model.layers)*0.3):]:
        layer.trainable = True
    
    model.compile(optimizer=keras.optimizers.Adam(1e-5), 
                  loss='categorical_crossentropy', metrics=['accuracy'])
    model.fit(train_ds, validation_data=val_ds, epochs=epochs_finetune, callbacks=callbacks, verbose=1)
    
    
    # Оценка на тестовой выборке
    y_true_onehot = np.concatenate([y for x, y in test_ds], axis=0)
    y_pred_probs = model.predict(test_ds)
    y_true = np.argmax(y_true_onehot, axis=1)
    y_pred = np.argmax(y_pred_probs, axis=1)
    
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    cm = confusion_matrix(y_true, y_pred)
    
    keras.backend.clear_session()
    
    return {
        'Network': model_name,
        'Split': split_name,
        'Accuracy': report['accuracy'],
        'Precision': report['macro avg']['precision'],
        'Recall': report['macro avg']['recall'],
        'F1': report['macro avg']['f1-score'],
        'Confusion_Matrix': cm.tolist()
    }

def main():
    prepare_data_splits()
    
    models = ['Xception', 'ResNet50V2', 'InceptionResNetV2', 'DenseNet201', 'NASNetLarge']
    splits = ['100_15_20', '250_30_20', '400_60_20', '700_100_20']
    
    results = []
    for split in splits:
        for model in models:
            print(f"[INFO] Training {model} on {split}")
            results.append(run_training(model, split))
            
    df = pd.DataFrame(results)
    df.to_excel('lab12_results.xlsx', index=False)
    
    # Вывод топ-3 моделей по F1-мере
    top3 = df.nlargest(3, 'F1')[['Network', 'Split', 'F1']]
    print("\n[INFO] Top 3 models by F1-Score:")
    print(top3.to_string(index=False))
    print(f"\n[INFO] Results saved to lab12_results.xlsx")

if __name__ == '__main__':
    main()

[INFO] Training Xception on 100_15_20
Found 1715 files belonging to 4 classes.
Found 326 files belonging to 4 classes.
Found 435 files belonging to 4 classes.
Epoch 1/5
108/108 [==============================] - 21s 128ms/step - loss: 14.9225 - accuracy: 0.2525 - val_loss: 1.3859 - val_accuracy: 0.2607 - lr: 0.0010
Epoch 2/5
108/108 [==============================] - 13s 120ms/step - loss: 1.3862 - accuracy: 0.2513 - val_loss: 1.3855 - val_accuracy: 0.2607 - lr: 0.0010
Epoch 3/5
108/108 [==============================] - 15s 139ms/step - loss: 1.3859 - accuracy: 0.2484 - val_loss: 1.3852 - val_accuracy: 0.2607 - lr: 0.0010
Epoch 4/5
108/108 [==============================] - 13s 118ms/step - loss: 1.3858 - accuracy: 0.2496 - val_loss: 1.3850 - val_accuracy: 0.2607 - lr: 0.0010
Epoch 5/5
108/108 [==============================] - 12s 108ms/step - loss: 1.3857 - accuracy: 0.2496 - val_loss: 1.3848 - val_accuracy: 0.2607 - lr: 0.0010
Epoch 1/5
108/108 [==============================] - 16